# TenderWin · Лід і мейл генератор v1.4 — ТЕСТОВИЙ РЕЖИМ

Вісім клітинок, згори вниз. Код редагувати не треба.

**Що робить:** шукає в Prozorro відхилення за поточну добу по всіх галузях і
всіх закупівлях, складає перший лист (V9) і робочу картку закупівлі, а для
кожного відхилення — **службову картку і протокол (рішення) замовника** у
`Мій диск / 0 Cases / <ID закупівлі> <ЄДРПОУ або ІПН учасника>`
(на компʼютері це `G:\Мій диск\0 Cases`).

**Чого не робить:** не завантажує документи пропозиції й тендерну
документацію, не аналізує їх, не пише скарг і **не надсилає нічого
клієнтам** — тільки на ваші тестові скриньки.

Потрібні чотири файли: `tenderwin_lead_mail.py`, `tenderwin_lead_engine.py`,
`lead_machine_v1.py`, `tenderwin_service_cards.py`. Якщо їх ще немає на
Диску — клітинка 2 запропонує завантажити їх із компʼютера.

## КЛІТИНКА 1 · Встановлення бібліотек

In [ ]:
!pip -q install google-api-python-client google-auth google-auth-oauthlib python-docx requests pdfplumber pypdf openpyxl
print("Готово. Далі — клітинка 2.")

## КЛІТИНКА 2 · Google Диск і чотири файли

Монтує Диск, знаходить теку `TenderWin` (будь-який регістр) і перевіряє файли.
Якщо якогось бракує — одразу відкриється вікно вибору файлів на компʼютері.

Потрібні: `tenderwin_lead_mail.py`, `tenderwin_lead_engine.py`,
`lead_machine_v1.py`, `tenderwin_service_cards.py` (службові картки).

In [ ]:
from google.colab import drive
import os, re, shutil

drive.mount('/content/drive')

КОРІНЬ = '/content/drive/MyDrive'
ІМЕНА_ТЕКИ = ('tenderwin',)          # будь-який регістр: TENDERWIN, TenderWin…
ПОТРІБНІ = ['tenderwin_lead_mail.py', 'tenderwin_lead_engine.py',
            'lead_machine_v1.py', 'tenderwin_service_cards.py']
ОЗНАКИ = ['.gmail_token.json', 'leads.db', 'tenderwin_lead_mail.db']


def вміст(тека):
    try:
        return set(os.listdir(тека))
    except OSError:
        return set()


def вага(тека):
    """Що більше в теці нашого — то вона і є справжня."""
    файли = вміст(тека)
    return (sum(1 for н in ПОТРІБНІ if н in файли),
            sum(1 for н in ОЗНАКИ if н in файли))


кандидати = sorted(
    (os.path.join(КОРІНЬ, назва) for назва in os.listdir(КОРІНЬ)
     if назва.lower() in ІМЕНА_ТЕКИ and os.path.isdir(os.path.join(КОРІНЬ, назва))),
    key=вага, reverse=True)

if кандидати:
    DYSK = кандидати[0]
    if len(кандидати) > 1:
        print("УВАГА: на Диску кілька тек TenderWin:")
        for тека in кандидати:
            є_файлів, є_ознак = вага(тека)
            print(f"   {тека}  (наших файлів: {є_файлів}, бази й токен: {є_ознак})")
        print("   Працюю з першою. Зайву можна видалити руками — але спершу")
        print("   перевірте, що в ній нічого потрібного не лишилось.")
else:
    DYSK = os.path.join(КОРІНЬ, 'TENDERWIN')
    os.makedirs(DYSK, exist_ok=True)
    print("Теки TenderWin не було — створив:", DYSK)

print("Робоча тека:", DYSK)


def чого_бракує():
    файли = вміст(DYSK)
    return [назва for назва in ПОТРІБНІ if назва not in файли]


def справжня_назва(назва):
    # Браузер часто додає "(1)": tenderwin_lead_mail (1).py
    чиста = re.sub(r"\s*\(\d+\)(?=\.py$)", "", назва).strip()
    return чиста if чиста in ПОТРІБНІ else назва


бракує = чого_бракує()
ГОТОВО = not бракує

if ГОТОВО:
    print("Усі файли на місці. Далі — клітинка 3.")
else:
    print("Бракує:", ", ".join(бракує))
    print("Оберіть ці файли на компʼютері у вікні нижче.")
    print("(Або покладіть їх у цю теку на Диску і виконайте клітинку ще раз.)")
    try:
        from google.colab import files
        завантажені = files.upload()
        for тимчасова in list(завантажені):
            назва = справжня_назва(тимчасова)
            if назва in ПОТРІБНІ:
                куди = os.path.join(DYSK, назва)
                if os.path.exists(куди):
                    os.remove(куди)
                shutil.move(тимчасова, куди)
                print("  покладено на Диск:", назва)
            else:
                print("  пропущено (не потрібен тут):", тимчасова)
    except Exception as помилка:
        print("Вікно вибору не відкрилося:", type(помилка).__name__)

    бракує = чого_бракує()
    ГОТОВО = not бракує
    print()
    print("Усі файли на місці. Далі — клітинка 3." if ГОТОВО
          else "Досі бракує: " + ", ".join(бракує) + ". Виконайте клітинку ще раз.")

## КЛІТИНКА 3 · Доступ до пошти

Секрет `GMAIL_OAUTH_JSON` має бути в Colab Secrets (ключик ліворуч), доступ
для цього ноутбука — увімкнений. У код секрети не вставляються ніколи.

In [ ]:
import sys, os

if "DYSK" not in globals():
    print("Спершу виконайте клітинку 2 — вона монтує Диск.")
    ФАЙЛИ_Є = False
else:
    if DYSK not in sys.path:
        sys.path.insert(0, DYSK)
    try:
        import tenderwin_lead_engine as E
        ФАЙЛИ_Є = True
    except ModuleNotFoundError as помилка:
        ФАЙЛИ_Є = False
        print("Немає файла:", (помилка.name or "?") + ".py")
        print("Поверніться до клітинки 2 — вона покладе файли на Диск.")

if ФАЙЛИ_Є:
    # Без цього рядка токен, що лежить файлом на Диску, просто не видно:
    # движок шукає .gmail_token.json у теці, яку йому назвали.
    E.TOKEN_DIR = DYSK
    у_теці = os.path.exists(os.path.join(DYSK, E.TOKEN_FILE_NAME))

    if E._get_secret(E.GMAIL_OAUTH_ENV):
        print("Доступ до пошти є" + (f" (файл {E.TOKEN_FILE_NAME} у теці на Диску)."
                                     if у_теці else " (секрет Colab)."))
        бракує_дозволів = E.missing_scopes()
        if бракує_дозволів:
            print("  Але токену бракує дозволів:", ", ".join(бракує_дозволів))
            print("  Зробіть E.reset_gmail_token(DYSK), потім E.mint_oauth_token().")
        else:
            print("Далі — клітинка 4.")
        if у_теці:
            print("  ⓘ Токен у теці на Диску може прочитати кожен, хто має")
            print("    доступ до цієї теки. Надійніше тримати його в Colab Secrets.")
    else:
        print(f"Доступу до пошти немає: ні секрету {E.GMAIL_OAUTH_ENV},")
        print(f"ні файла {E.TOKEN_FILE_NAME} у теці {DYSK}.")
        print("Без нього працює сухий прогін (клітинка 6), відправки не буде.")
        print("Як отримати токен: E.mint_oauth_token()")

## КЛІТИНКА 4 · Завантаження скрипта

In [ ]:
import importlib

M = None
if "DYSK" not in globals():
    print("Спершу виконайте клітинку 2.")
else:
    try:
        import tenderwin_lead_mail as M
        importlib.reload(M)
    except ModuleNotFoundError as помилка:
        M = None
        print("Немає файла:", (помилка.name or "?") + ".py")
        print("Поверніться до клітинки 2.")

if M is not None:
    M.setup(DYSK)      # база, білий список адрес, історія попереднього движка

## КЛІТИНКА 5 · Налаштування

Змінювати не обовʼязково. Значення нижче — чинне рішення власника.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.LETTER_VERSION = "V9_OWNER"       # ЧИННИЙ лист (рішення 06.10.2026)
    M.LETTER_FORMAT  = "HTML_AND_TEXT"  # "TEXT_ONLY" — лист лише текстом
    M.WINDOW_MODE    = "TODAY_ONLY"     # лише поточна доба
    M.SKIP_WITHOUT_COMPLAINT_ROUTE = True   # без строку оскарження — не пишемо
    M.MIN_LOT_VALUE  = 0                # 0 = без порогу вартості
    M.SERVICE_CARDS  = True             # службові картки з протоколами
    M.SERVICE_CARDS_FRESH = True        # свіжий стан рішення з Prozorro
    M.CASES_DIR = ""                    # "" = «Мій диск/0 Cases» (G:\Мій диск\0 Cases)

    print("режим:", M.MODE, "· лист:", M.LETTER_VERSION, "·", M.LETTER_FORMAT)
    print("адрес для тестової доставки:", len(M.TEST_RECIPIENTS))
    print("службові картки:", "так" if M.SERVICE_CARDS else "ні")
    if M.SERVICE_CARDS:
        print("справи (картка + протокол):", M.cases_root())

## КЛІТИНКА 6 · Сухий прогін

Шукає відхилення за сьогодні, складає листи у файли і робочі картки.
**Нічого нікуди не надсилає.**

У теці `vyhid_lystiv/eml` біля кожного листа лежить файл `.html` — відкрийте
його у браузері, щоб побачити лист таким, яким його побачить клієнт.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    результат = M.go(dysk=DYSK)

## КЛІТИНКА 7 · Відправка на ваші скриньки і стан бази

Листи підуть **тільки** на адреси зі списку `TEST_RECIPIENTS`. База це
перевіряє окремо від коду: адресу клієнта туди не пропустить тригер.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.send(dysk=DYSK)     # відправка на тестові скриньки + звірка з Gmail
    M.state(dysk=DYSK)    # що зараз у базі

## КЛІТИНКА 8 · Службові картки ще раз (за потреби)

Після `M.go()` картки вже зроблені: кожна справа — окрема тека в
`Мій диск / 0 Cases` з назвою «ID закупівлі + ЄДРПОУ (ІПН) учасника», у ній
службова картка і протокол (рішення) замовника. Ця клітинка потрібна лише
тоді, коли крок карток обірвався або коли треба зібрати картки за інший день.

Уже завантажені файли вдруге не качаються. Картку, у якій ви щось дописали,
скрипт **не перезаписує** — нова версія ляже поруч окремим файлом.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.kartky(dysk=DYSK)                      # події останнього прогону
    # M.kartky(day="2026-10-02", dysk=DYSK)  # усі події конкретного дня

---
### Якщо щось пішло не так

| Повідомлення | Що робити |
|---|---|
| `Немає файла: …py` | клітинка 2 — покладіть файли на Диск |
| `TRANSPORT_UNAVAILABLE` | немає секрету `GMAIL_OAUTH_JSON` — клітинка 3 |
| `TEST_ALLOWLIST_EMPTY` | порожній `TEST_RECIPIENTS` — так задумано, заповніть |
| `PROZORRO_UNAVAILABLE` | Prozorro не відповідає — спробуйте за кілька хвилин |
| `Обхід стрічки: INCOMPLETE` | прочитано не всю стрічку; у звіті написано чому |
| `листи без певного стану` | зробіть `M.send(dysk=DYSK)` — він звірить їх із Gmail |
| `SERVICE_CARDS_MODULE_MISSING` | немає `tenderwin_service_cards.py` — клітинка 2; листи цим не зачеплені |
| `SERVICE_CARDS_FAILED` | крок карток обірвався; листи збережено. Повторіть `M.kartky(dysk=DYSK)` |

**Де службові картки і протоколи:** `Мій диск → 0 Cases → <UA-ID> <ЄДРПОУ/ІПН>`
(на компʼютері `G:\Мій диск\0 Cases`). Якщо того самого учасника в тій самій
закупівлі відхилили двічі (два лоти, повторне рішення), друга справа — у теці
з « (2)» в кінці.

**Перелік для відбору:** `TenderWin → Службові картки → <дата> →
_ПЕРЕЛІК_<прогін>.xlsx` — один рядок на подію, колонка «Тека справи».

Перевірити сам скрипт: `M.tests(DYSK)`